## Imports e Configurações

In [2]:
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt, resample_poly
from sklearn.model_selection import train_test_split

CLASSES = ["Present", "Unknown", "Absent"]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

@dataclass
class Config:
    data_dir: Path = Path("../data/circor")
    fs_target: int = 1000   # Hz (Nyquist = 500 Hz)
    band: tuple = (25, 400)        # passa-banda (Hz)
    win_sec: float = 5.0           # tamanho da janela (s)
    hop_sec: float = 2.5           # passo entre janelas (s)
    # --- log-mel / CNN 2D ---
    n_mels: int = 40
    frame_len: int = 128           # amostras (128 ms a 1 kHz)
    frame_step: int = 32           # amostras (32 ms)
    # --- RNN ---
    n_mfcc: int = 13
    rnn_frame_step: int = 64       # passo maior => sequências mais curtas (~77)
    seed: int = 42

    @property
    def wav_dir(self):
        return self.data_dir / "training_data"

    @property
    def csv_path(self):
        return self.data_dir / "training_data.csv"

CFG = Config()
print("CSV existe:", CFG.csv_path.exists(), "| pasta wav existe:", CFG.wav_dir.exists())

CSV existe: True | pasta wav existe: True


## Metadados e split por paciente

O split é feito **por paciente** (todas as gravações de um paciente ficam no mesmo subconjunto) e **estratificado** pela classe `Murmur`. O resultado é guardado em `splits.json`, que deve ser commitado para todo o grupo usar o mesmo split.

In [5]:
def load_metadata(cfg: Config = CFG) -> pd.DataFrame:
    """Carrega o arquivo CSV com os metadados dos áudios."""
    df = pd.read_csv(cfg.csv_path)[["Patient ID", "Murmur"]].copy()
    df.columns = ["pid", "murmur"]
    df["pid"] = df["pid"].astype(str)
    df["y"] = df["murmur"].map(CLASS2IDX)
    assert df["y"].notna().all(), "Valores de Murmur inesperados"
    return df

def make_splits(meta: pd.DataFrame, val_size=0.15, test_size=0.15, seed=42):
    """Split estratificado POR PACIENTE (uma linha do meta = um paciente)."""
    pids, ys = meta["pid"].values, meta["y"].values
    trval, test = train_test_split(pids, test_size=test_size, stratify=ys, random_state=seed)
    y_traval = meta.set_index("pid").loc[trval, "y"].values
    train, val = train_test_split(trval, test_size=val_size / (1 - test_size), stratify=y_traval, random_state=seed)
    return {"train": list(train), "val": list(val), "test": list(test)}

def get_splits(meta, path="splits.json", seed=42):
    """Carrega o split guardado ou cria-o (todos no grupo devem usar o mesmo)."""
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    splits = make_splits(meta, seed=seed)
    path.write_text(json.dumps(splits))
    return splits

In [6]:
meta = load_metadata()
splits = get_splits(meta)

print(meta["murmur"].value_counts(normalize=True).round(3), "\n")
for k, v in splits.items():
    print(k, len(v), meta[meta.pid.isin(v)].murmur.value_counts().to_dict())

# Garantia: sem pacientes repetidos entre subconjuntos
assert not (set(splits["train"]) & set(splits["val"]))
assert not (set(splits["train"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))

murmur
Absent     0.738
Present    0.190
Unknown    0.072
Name: proportion, dtype: float64 

train 658 {'Absent': 485, 'Present': 125, 'Unknown': 48}
val 142 {'Absent': 105, 'Present': 27, 'Unknown': 10}
test 142 {'Absent': 105, 'Present': 27, 'Unknown': 10}
